In [1]:
# ============================================================
# M0 — CORRECTED BASELINE RECALCULATION
# CELL 1: CONFIGURATION AND IMPORTS
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# ACCOUNT CONFIGURATION
# ------------------------------------------------------------

# ACCOUNT A:
ACCOUNT_ROLE = "A_REFERENCE_MODEL"

# ACCOUNT B:
# ACCOUNT_ROLE = "B_BASE_RAW_402"

SEED = 42
N_SPLITS = 5
N_REPEATS = 5

CLASS_ORDER = ["N", "S", "V", "F", "Q"]
FOUR_CLASS_ORDER = ["N", "S", "V", "Q"]

EXPECTED_COUNTS = {
    "N": 90341,
    "S": 2781,
    "V": 7229,
    "F": 802,
    "Q": 3894
}

EXPECTED_BEATS = 105047
EXPECTED_RECORDS = 46
EXPECTED_PEOPLE = 45

BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_SEED = 20261105

EXPECTED_SPLIT_FINGERPRINT = (
    "5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9"
)

if ACCOUNT_ROLE == "A_REFERENCE_MODEL":
    EXPERIMENT = "REFERENCE_MODEL"
    TREES = 200
    MAX_FEATURES = "log2"
    FEATURE_COUNT = 12

elif ACCOUNT_ROLE == "B_BASE_RAW_402":
    EXPERIMENT = "BASE_RAW_402"
    TREES = 100
    MAX_FEATURES = "sqrt"
    FEATURE_COUNT = 402

else:
    raise ValueError("Invalid ACCOUNT_ROLE.")


OUTPUT_DIR = Path(
    f"/kaggle/working/ECG_Project/02_M0_Corrected/{EXPERIMENT}"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Account:", ACCOUNT_ROLE)
print("Experiment:", EXPERIMENT)
print("Random Forest trees:", TREES)
print("Maximum features:", MAX_FEATURES)
print("Feature count:", FEATURE_COUNT)
print("Output folder:", OUTPUT_DIR)

print("\nNo new splits will be generated.")
print("Original M0 results will not be overwritten.")

Account: A_REFERENCE_MODEL
Experiment: REFERENCE_MODEL
Random Forest trees: 200
Maximum features: log2
Feature count: 12
Output folder: /kaggle/working/ECG_Project/02_M0_Corrected/REFERENCE_MODEL

No new splits will be generated.
Original M0 results will not be overwritten.


In [3]:
# ============================================================
# CELL 2: LOAD CORRECTED DATASET — FIXED VERSION
# ============================================================

INPUT_ROOT = Path("/kaggle/input")


def find_exactly_one(filename):

    matches = sorted(INPUT_ROOT.rglob(filename))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; "
            f"found {len(matches)}: {[str(p) for p in matches]}"
        )

    return matches[0]


# ------------------------------------------------------------
# 1. Find the exact corrected CSV
# ------------------------------------------------------------

corrected_path = find_exactly_one(
    "MITBIH_MLII_DualLead_Dataset_corrected.csv"
)

print("Using corrected dataset:", corrected_path)


# ------------------------------------------------------------
# 2. Load the corrected dataset
# ------------------------------------------------------------

data = pd.read_csv(
    corrected_path,
    low_memory=False
)

data.columns = [
    str(c).strip().strip('"').strip("'").strip()
    for c in data.columns
]


# IMPORTANT:
# Check original dimensions BEFORE adding helper columns.

original_loaded_shape = data.shape

print("\nORIGINAL LOADED DATASET SHAPE:")
print(original_loaded_shape)

if original_loaded_shape != (106870, 404):

    raise ValueError(
        f"Unexpected original dataset shape: "
        f"{original_loaded_shape}. Expected (106870, 404)."
    )


# ------------------------------------------------------------
# 3. Verify all 402 waveform columns
# ------------------------------------------------------------

missing_columns = [
    c for c in SIGNAL_COLS
    if c not in data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing ECG signal columns: {missing_columns}"
    )

assert len(SIGNAL_COLS) == 402


# ------------------------------------------------------------
# 4. Prepare Record_ID and original annotation labels
# ------------------------------------------------------------

data["Record_ID"] = pd.to_numeric(
    data["Record_ID"],
    errors="raise"
).astype(int)

data["Label"] = (
    data["Label"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 5. Create AAMI labels
# ------------------------------------------------------------

data["AAMI_Label"] = (
    data["Label"].map(AAMI_MAP)
)


# ------------------------------------------------------------
# 6. Preserve original physical row positions
# ------------------------------------------------------------

data["_Source_Row_Index"] = np.arange(
    len(data),
    dtype=np.int64
)


# Candidate 1:
# Original M0 may have created Row_Index before AAMI filtering.

candidate_source_row_ids = (
    data.loc[
        data["AAMI_Label"].isin(CLASS_ORDER),
        "_Source_Row_Index"
    ].to_numpy(dtype=np.int64)
)


# Candidate 2:
# Original M0 may have created Row_Index after filtering.

candidate_filtered_row_ids = np.arange(
    int(data["AAMI_Label"].isin(CLASS_ORDER).sum()),
    dtype=np.int64
)


# ------------------------------------------------------------
# 7. Keep only the five AAMI classes
# ------------------------------------------------------------

aami_df = data[
    data["AAMI_Label"].isin(CLASS_ORDER)
].copy().reset_index(drop=True)


# ------------------------------------------------------------
# 8. Assign Person_ID
# ------------------------------------------------------------

aami_df["Person_ID"] = [
    "person_201_202"
    if rid in (201, 202)
    else f"person_{rid}"
    for rid in aami_df["Record_ID"]
]


# ------------------------------------------------------------
# 9. Validate dataset
# ------------------------------------------------------------

observed_counts = (
    aami_df["AAMI_Label"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
    .to_dict()
)

assert len(aami_df) == EXPECTED_BEATS
assert aami_df["Record_ID"].nunique() == EXPECTED_RECORDS
assert aami_df["Person_ID"].nunique() == EXPECTED_PEOPLE

assert observed_counts == EXPECTED_COUNTS, observed_counts

assert aami_df["Record_ID"].isin([201, 202]).any()

assert np.isfinite(
    aami_df[SIGNAL_COLS].to_numpy(dtype=np.float32)
).all()


# ------------------------------------------------------------
# 10. Display validation results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CORRECTED DATASET VALIDATION")
print("=" * 65)

print("Original dataset shape:", original_loaded_shape)
print("Valid heartbeat count:", len(aami_df))
print("Recordings:", aami_df["Record_ID"].nunique())
print("People:", aami_df["Person_ID"].nunique())

print("\nClass counts:")
print(observed_counts)

print("\nOriginal row-index candidates:")
print("Before filtering:", len(candidate_source_row_ids))
print("After filtering:", len(candidate_filtered_row_ids))

print("\nFeature columns:", len(SIGNAL_COLS))

print("\nCELL 2 PASS")

Using corrected dataset: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/MITBIH_MLII_DualLead_Dataset_corrected.csv

ORIGINAL LOADED DATASET SHAPE:
(106870, 404)

CORRECTED DATASET VALIDATION
Original dataset shape: (106870, 404)
Valid heartbeat count: 105047
Recordings: 46
People: 45

Class counts:
{'N': 90341, 'S': 2781, 'V': 7229, 'F': 802, 'Q': 3894}

Original row-index candidates:
Before filtering: 105047
After filtering: 105047

Feature columns: 402

CELL 2 PASS


In [4]:
# ============================================================
# CELL 3: RESTORE ORIGINAL M0 SPLITS — FIXED VERSION
# Supports CSV and CSV.GZ
# ============================================================

# ------------------------------------------------------------
# 1. Find split files regardless of compression
# ------------------------------------------------------------

def find_split_file(possible_names):

    matches = []

    for path in INPUT_ROOT.rglob("*"):

        if path.is_file() and path.name.lower() in [
            name.lower() for name in possible_names
        ]:
            matches.append(path)

    if len(matches) != 1:

        raise FileNotFoundError(
            f"Expected exactly one of {possible_names}. "
            f"Found {len(matches)}: {[str(p) for p in matches]}"
        )

    return matches[0]


row_fold_path = find_split_file([
    "m0_row_fold_membership.csv.gz",
    "m0_row_fold_membership.csv"
])

person_fold_path = find_split_file([
    "m0_person_fold_membership.csv"
])

fingerprint_path = find_split_file([
    "m0_split_fingerprint.txt"
])


print("ORIGINAL M0 SPLIT FILES")
print("Row membership:", row_fold_path)
print("Person membership:", person_fold_path)
print("Fingerprint:", fingerprint_path)


# ------------------------------------------------------------
# 2. Load the split files
# ------------------------------------------------------------

# compression="infer" automatically handles CSV and CSV.GZ.
row_fold = pd.read_csv(
    row_fold_path,
    compression="infer"
)

person_fold = pd.read_csv(
    person_fold_path,
    compression="infer"
)

row_fold["Row_Index"] = pd.to_numeric(
    row_fold["Row_Index"],
    errors="raise"
).astype(np.int64)

row_fold["Repeat"] = pd.to_numeric(
    row_fold["Repeat"],
    errors="raise"
).astype(int)

row_fold["Fold"] = pd.to_numeric(
    row_fold["Fold"],
    errors="raise"
).astype(int)

person_fold["Repeat"] = pd.to_numeric(
    person_fold["Repeat"],
    errors="raise"
).astype(int)

person_fold["Fold"] = pd.to_numeric(
    person_fold["Fold"],
    errors="raise"
).astype(int)

person_fold["Person_ID"] = (
    person_fold["Person_ID"].astype(str)
)


print("\nSPLIT FILE DIMENSIONS")
print("Row membership:", row_fold.shape)
print("Person membership:", person_fold.shape)


# ------------------------------------------------------------
# 3. Verify original split fingerprint
# ------------------------------------------------------------

computed_fingerprint = hashlib.sha256(
    person_fold
    .sort_values(["Repeat", "Person_ID"])
    .to_csv(
        index=False,
        lineterminator="\n"
    )
    .encode("utf-8")
).hexdigest()

saved_fingerprint = fingerprint_path.read_text().strip()

print("\nSPLIT FINGERPRINT")
print("Saved:   ", saved_fingerprint)
print("Computed:", computed_fingerprint)
print("Expected:", EXPECTED_SPLIT_FINGERPRINT)


assert saved_fingerprint == EXPECTED_SPLIT_FINGERPRINT

assert computed_fingerprint == EXPECTED_SPLIT_FINGERPRINT


# ------------------------------------------------------------
# 4. Identify the original Row_Index convention
# ------------------------------------------------------------

saved_row_ids = set(
    row_fold["Row_Index"].unique()
)

source_id_set = set(
    candidate_source_row_ids.tolist()
)

filtered_id_set = set(
    candidate_filtered_row_ids.tolist()
)

if saved_row_ids == source_id_set:

    aami_df["Row_Index"] = candidate_source_row_ids

    ROW_INDEX_MODE = "Created before AAMI filtering"

elif saved_row_ids == filtered_id_set:

    aami_df["Row_Index"] = candidate_filtered_row_ids

    ROW_INDEX_MODE = "Created after AAMI filtering"

else:

    raise ValueError(
        "Saved Row_Index values do not match either "
        "original indexing convention. Stop; do not train."
    )


print("\nOriginal Row_Index convention:", ROW_INDEX_MODE)


# ------------------------------------------------------------
# 5. Validate split dimensions and uniqueness
# ------------------------------------------------------------

assert len(row_fold) == EXPECTED_BEATS * N_REPEATS

assert not row_fold.duplicated(
    ["Repeat", "Row_Index"]
).any()

assert len(person_fold) == EXPECTED_PEOPLE * N_REPEATS

assert not person_fold.duplicated(
    ["Repeat", "Person_ID"]
).any()

assert set(person_fold["Person_ID"]) == set(
    aami_df["Person_ID"]
)


# ------------------------------------------------------------
# 6. Check every repeat has all 105,047 rows
# ------------------------------------------------------------

for repeat in range(1, N_REPEATS + 1):

    current = row_fold[
        row_fold["Repeat"] == repeat
    ]

    assert len(current) == EXPECTED_BEATS

    assert set(current["Row_Index"]) == saved_row_ids

    assert set(current["Fold"]) == {1, 2, 3, 4, 5}


# ------------------------------------------------------------
# 7. Verify person assignments against row assignments
# ------------------------------------------------------------

row_person_map = aami_df.set_index(
    "Row_Index"
)["Person_ID"]

row_fold["Person_ID"] = row_fold["Row_Index"].map(
    row_person_map
)

assert row_fold["Person_ID"].notna().all()

joined = row_fold.merge(
    person_fold,
    on=["Repeat", "Person_ID"],
    how="left",
    suffixes=("_row", "_person"),
    validate="many_to_one"
)

assert joined["Fold_person"].notna().all()

assert (
    joined["Fold_row"] == joined["Fold_person"]
).all()


# ------------------------------------------------------------
# 8. Build positional fold assignments
# ------------------------------------------------------------

row_to_position = {
    int(row_id): position
    for position, row_id in enumerate(
        aami_df["Row_Index"].to_numpy()
    )
}

row_fold["Position"] = row_fold["Row_Index"].map(
    row_to_position
)

assert row_fold["Position"].notna().all()

row_fold["Position"] = row_fold[
    "Position"
].astype(int)


fold_assignment_by_repeat = {}

for repeat in range(1, N_REPEATS + 1):

    current = row_fold[
        row_fold["Repeat"] == repeat
    ]

    assignments = np.full(
        EXPECTED_BEATS,
        -1,
        dtype=np.int8
    )

    assignments[
        current["Position"].to_numpy()
    ] = current["Fold"].to_numpy()

    assert np.all(assignments > 0)

    fold_assignment_by_repeat[repeat] = assignments


# ------------------------------------------------------------
# 9. Verify no person or recording crosses folds
# ------------------------------------------------------------

for repeat in range(1, N_REPEATS + 1):

    current = row_fold[
        row_fold["Repeat"] == repeat
    ]

    assert current.groupby(
        "Person_ID"
    )["Fold"].nunique().max() == 1

    record_check = current.merge(
        aami_df[["Row_Index", "Record_ID"]],
        on="Row_Index",
        validate="many_to_one"
    )

    assert record_check.groupby(
        "Record_ID"
    )["Fold"].nunique().max() == 1


# ------------------------------------------------------------
# 10. Final results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ORIGINAL M0 SPLIT VERIFICATION")
print("=" * 65)

print("Row membership rows:", len(row_fold))
print("Person membership rows:", len(person_fold))
print("Repeats:", N_REPEATS)
print("Folds per repeat:", N_SPLITS)

print("Original split fingerprint:", computed_fingerprint)

print("Original Row_Index convention:", ROW_INDEX_MODE)

print("Person-level leakage: None")
print("Record-level leakage: None")
print("New splits generated: NO")

print("\nCELL 3 PASS — ORIGINAL SPLITS RESTORED.")

ORIGINAL M0 SPLIT FILES
Row membership: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_row_fold_membership.csv
Person membership: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_person_fold_membership.csv
Fingerprint: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_split_fingerprint.txt

SPLIT FILE DIMENSIONS
Row membership: (525235, 3)
Person membership: (225, 3)

SPLIT FINGERPRINT
Saved:    5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
Computed: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
Expected: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9

Original Row_Index convention: Created after AAMI filtering

ORIGINAL M0 SPLIT VERIFICATION
Row membership rows: 525235
Person membership rows: 225
Repeats: 5
Folds per repeat: 5
Original split fingerprint: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
Original Row_Index conventio

In [5]:
# ============================================================
# DEFINE ALL 402 ECG SIGNAL COLUMNS
# ============================================================

SIGNAL_COLS = (
    [f"MLII_{i}" for i in range(201)]
    +
    [f"Lead2_{i}" for i in range(201)]
)

assert len(SIGNAL_COLS) == 402

print("ECG signal columns:", len(SIGNAL_COLS))
print("First columns:", SIGNAL_COLS[:5])
print("Last columns:", SIGNAL_COLS[-5:])

print("PASS: All 402 signal columns identified.")

ECG signal columns: 402
First columns: ['MLII_0', 'MLII_1', 'MLII_2', 'MLII_3', 'MLII_4']
Last columns: ['Lead2_196', 'Lead2_197', 'Lead2_198', 'Lead2_199', 'Lead2_200']
PASS: All 402 signal columns identified.


In [6]:
# ============================================================
# CELL 4: FEATURE ENGINEERING
# ============================================================

y = aami_df["AAMI_Label"].to_numpy(dtype=str)
groups = aami_df["Person_ID"].to_numpy(dtype=str)
record_ids = aami_df["Record_ID"].to_numpy(dtype=np.int64)
row_ids = aami_df["Row_Index"].to_numpy(dtype=np.int64)

if ACCOUNT_ROLE == "A_REFERENCE_MODEL":

    print("Building REFERENCE_MODEL features...")

    # Extract MLII only.
    wave = aami_df[
        [f"MLII_{i}" for i in range(201)]
    ].to_numpy(dtype=np.float32)

    # Per-beat normalisation.
    means = wave.mean(axis=1, keepdims=True)
    stds = wave.std(axis=1, keepdims=True)

    stds[stds == 0] = 1.0

    normalized = (wave - means) / stds

    # First and second derivatives.
    d1 = np.diff(normalized, axis=1)
    d2 = np.diff(normalized, n=2, axis=1)

    # Distribution statistics.
    q25, q50, q75 = np.percentile(
        normalized,
        [25, 50, 75],
        axis=1
    )

    X = np.column_stack([
        d1.mean(axis=1),
        d1.std(axis=1),
        d1.max(axis=1),
        d1.min(axis=1),
        np.abs(d1).mean(axis=1),
        np.square(d1).mean(axis=1),
        d2.std(axis=1),
        np.abs(d2).mean(axis=1),
        q50,
        q75 - q25,
        q25,
        q75
    ]).astype(np.float32)


else:

    print("Building BASE_RAW_402 features...")

    X = aami_df[
        SIGNAL_COLS
    ].to_numpy(dtype=np.float32)


# ------------------------------------------------------------
# Feature validation
# ------------------------------------------------------------

assert X.shape == (
    EXPECTED_BEATS,
    FEATURE_COUNT
)

assert np.isfinite(X).all()

print("\nFEATURE MATRIX")
print("Experiment:", EXPERIMENT)
print("Shape:", X.shape)
print("Finite values:", np.isfinite(X).all())

print("\nCELL 4 PASS")

Building REFERENCE_MODEL features...

FEATURE MATRIX
Experiment: REFERENCE_MODEL
Shape: (105047, 12)
Finite values: True

CELL 4 PASS


In [7]:
# ============================================================
# CELL 5: TRAIN THE CORRECTED BASELINE
# ============================================================

prediction_parts = []
fold_run_rows = []

for repeat in range(1, N_REPEATS + 1):

    fold_assignment = fold_assignment_by_repeat[repeat]

    for fold in range(1, N_SPLITS + 1):

        val_idx = np.flatnonzero(
            fold_assignment == fold
        )

        train_idx = np.flatnonzero(
            fold_assignment != fold
        )

        assert len(val_idx) > 0
        assert len(train_idx) > 0

        # Verify no person or recording leakage.
        assert set(groups[train_idx]).isdisjoint(
            set(groups[val_idx])
        )

        assert set(record_ids[train_idx]).isdisjoint(
            set(record_ids[val_idx])
        )

        # Original model configuration.
        model = RandomForestClassifier(
            n_estimators=TREES,
            max_features=MAX_FEATURES,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=SEED,
            n_jobs=-1,
            class_weight=None
        )

        model.fit(
            X[train_idx],
            y[train_idx]
        )

        y_pred = model.predict(
            X[val_idx]
        )

        raw_prob = model.predict_proba(
            X[val_idx]
        )

        # Preserve a fixed N, S, V, F, Q probability order.
        probabilities = np.zeros(
            (len(val_idx), len(CLASS_ORDER)),
            dtype=np.float32
        )

        for j, class_name in enumerate(model.classes_):

            probabilities[
                :, CLASS_ORDER.index(str(class_name))
            ] = raw_prob[:, j]


        part = pd.DataFrame({
            "Experiment": EXPERIMENT,
            "Repeat": repeat,
            "Fold": fold,
            "Row_Index": row_ids[val_idx],
            "Record_ID": record_ids[val_idx],
            "Person_ID": groups[val_idx],
            "True_Label": y[val_idx],
            "Predicted_Label": y_pred
        })

        for j, class_name in enumerate(CLASS_ORDER):
            part[f"Prob_{class_name}"] = probabilities[:, j]

        prediction_parts.append(part)

        fold_run_rows.append({
            "Experiment": EXPERIMENT,
            "Repeat": repeat,
            "Fold": fold,
            "Training_Beats": len(train_idx),
            "Validation_Beats": len(val_idx),
            "Training_People": len(set(groups[train_idx])),
            "Validation_People": len(set(groups[val_idx])),
            "Training_Records": len(set(record_ids[train_idx])),
            "Validation_Records": len(set(record_ids[val_idx]))
        })

        print(
            f"Completed {EXPERIMENT}: "
            f"Repeat {repeat}/{N_REPEATS}, "
            f"Fold {fold}/{N_SPLITS}"
        )


# ------------------------------------------------------------
# Combine predictions
# ------------------------------------------------------------

predictions = pd.concat(
    prediction_parts,
    ignore_index=True
)

fold_run_audit = pd.DataFrame(fold_run_rows)

assert len(predictions) == EXPECTED_BEATS * N_REPEATS

assert not predictions.duplicated(
    ["Repeat", "Row_Index"]
).any()

assert predictions.groupby(
    "Repeat"
)["Row_Index"].nunique().eq(EXPECTED_BEATS).all()


# Probability validation.
prob_cols = [f"Prob_{c}" for c in CLASS_ORDER]

probability_sums = predictions[
    prob_cols
].sum(axis=1).to_numpy()

assert np.allclose(
    probability_sums,
    1.0,
    atol=1e-5
)


# ------------------------------------------------------------
# Save OOF predictions
# ------------------------------------------------------------

pred_path = OUTPUT_DIR / (
    f"m0_corrected_oof_predictions_{EXPERIMENT}.csv.gz"
)

predictions.to_csv(
    pred_path,
    index=False,
    compression="gzip"
)

fold_run_audit.to_csv(
    OUTPUT_DIR / f"m0_corrected_fold_run_audit_{EXPERIMENT}.csv",
    index=False
)

print("\n" + "=" * 65)
print("MODEL TRAINING COMPLETE")
print("=" * 65)

print("Completed fold runs:", len(fold_run_audit))
print("OOF prediction rows:", len(predictions))
print("Expected OOF rows:", EXPECTED_BEATS * N_REPEATS)
print("Probability sums valid: PASS")
print("Saved:", pred_path)

Completed REFERENCE_MODEL: Repeat 1/5, Fold 1/5
Completed REFERENCE_MODEL: Repeat 1/5, Fold 2/5
Completed REFERENCE_MODEL: Repeat 1/5, Fold 3/5
Completed REFERENCE_MODEL: Repeat 1/5, Fold 4/5
Completed REFERENCE_MODEL: Repeat 1/5, Fold 5/5
Completed REFERENCE_MODEL: Repeat 2/5, Fold 1/5
Completed REFERENCE_MODEL: Repeat 2/5, Fold 2/5
Completed REFERENCE_MODEL: Repeat 2/5, Fold 3/5
Completed REFERENCE_MODEL: Repeat 2/5, Fold 4/5
Completed REFERENCE_MODEL: Repeat 2/5, Fold 5/5
Completed REFERENCE_MODEL: Repeat 3/5, Fold 1/5
Completed REFERENCE_MODEL: Repeat 3/5, Fold 2/5
Completed REFERENCE_MODEL: Repeat 3/5, Fold 3/5
Completed REFERENCE_MODEL: Repeat 3/5, Fold 4/5
Completed REFERENCE_MODEL: Repeat 3/5, Fold 5/5
Completed REFERENCE_MODEL: Repeat 4/5, Fold 1/5
Completed REFERENCE_MODEL: Repeat 4/5, Fold 2/5
Completed REFERENCE_MODEL: Repeat 4/5, Fold 3/5
Completed REFERENCE_MODEL: Repeat 4/5, Fold 4/5
Completed REFERENCE_MODEL: Repeat 4/5, Fold 5/5
Completed REFERENCE_MODEL: Repeat 5/5, F

In [8]:
# ============================================================
# CELL 6: POOLED-REPEAT MODEL METRICS
# ============================================================

metric_rows = []
class_rows = []
confusion_rows = []

for repeat in range(1, N_REPEATS + 1):

    part = predictions[
        predictions["Repeat"] == repeat
    ]

    yt = part["True_Label"].to_numpy()
    yp = part["Predicted_Label"].to_numpy()

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            yt,
            yp,
            labels=CLASS_ORDER,
            zero_division=0
        )
    )

    non_f = part[
        part["True_Label"].isin(FOUR_CLASS_ORDER)
    ]

    metrics = {
        "Experiment": EXPERIMENT,
        "Repeat": repeat,

        "Macro_F1_5_Class": f1_score(
            yt,
            yp,
            labels=CLASS_ORDER,
            average="macro",
            zero_division=0
        ),

        "Macro_F1_4_Class_N_S_V_Q": f1_score(
            non_f["True_Label"],
            non_f["Predicted_Label"],
            labels=FOUR_CLASS_ORDER,
            average="macro",
            zero_division=0
        ),

        "Balanced_Accuracy": balanced_accuracy_score(
            yt,
            yp
        ),

        "Accuracy": accuracy_score(
            yt,
            yp
        )
    }

    metric_rows.append(metrics)


    class_rows.append(
        pd.DataFrame({
            "Experiment": EXPERIMENT,
            "Repeat": repeat,
            "Class": CLASS_ORDER,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Support": support
        })
    )


    cm = confusion_matrix(
        yt,
        yp,
        labels=CLASS_ORDER
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{c}" for c in CLASS_ORDER],
        columns=[f"Predicted_{c}" for c in CLASS_ORDER]
    )

    cm_df.insert(
        0,
        "Actual_Class",
        cm_df.index
    )

    cm_df["Repeat"] = repeat

    confusion_rows.append(
        cm_df.reset_index(drop=True)
    )


# ------------------------------------------------------------
# Combine metric tables
# ------------------------------------------------------------

repeat_metrics = pd.DataFrame(metric_rows)

per_class = pd.concat(
    class_rows,
    ignore_index=True
)

confusions = pd.concat(
    confusion_rows,
    ignore_index=True
)


# ------------------------------------------------------------
# Headline: average of five pooled-repeat results
# ------------------------------------------------------------

headline = pd.DataFrame([{

    "Experiment": EXPERIMENT,

    "Headline_Macro_F1_5_Class":
        repeat_metrics["Macro_F1_5_Class"].mean(),

    "Macro_F1_4_Class_N_S_V_Q":
        repeat_metrics["Macro_F1_4_Class_N_S_V_Q"].mean(),

    "Balanced_Accuracy":
        repeat_metrics["Balanced_Accuracy"].mean(),

    "Accuracy":
        repeat_metrics["Accuracy"].mean()

}])


print("HEADLINE MODEL RESULTS")

display(headline)

print("\nMETRICS BY REPEAT")
display(repeat_metrics)

print("\nPER-CLASS METRICS")
display(per_class)

print("\nCONFUSION MATRICES")
display(confusions)


# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

repeat_metrics.to_csv(
    OUTPUT_DIR / f"m0_corrected_repeat_metrics_{EXPERIMENT}.csv",
    index=False
)

per_class.to_csv(
    OUTPUT_DIR / f"m0_corrected_per_class_metrics_{EXPERIMENT}.csv",
    index=False
)

confusions.to_csv(
    OUTPUT_DIR / f"m0_corrected_confusion_matrices_{EXPERIMENT}.csv",
    index=False
)

headline.to_csv(
    OUTPUT_DIR / f"m0_corrected_headline_summary_{EXPERIMENT}.csv",
    index=False
)

print("\nCELL 6 COMPLETE")

HEADLINE MODEL RESULTS


,Experiment,Headline_Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,0.43527,0.546823,0.466758,0.898567



METRICS BY REPEAT


,Experiment,Repeat,Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,1,0.518445,0.651288,0.535339,0.913344
1,REFERENCE_MODEL,2,0.316230,0.397515,0.364539,0.889211
2,REFERENCE_MODEL,3,0.520605,0.653948,0.540737,0.912877
3,REFERENCE_MODEL,4,0.312341,0.392648,0.362271,0.875903
4,REFERENCE_MODEL,5,0.508729,0.638717,0.530906,0.901501



PER-CLASS METRICS


,Experiment,Repeat,Class,Precision,Recall,F1,Support
0,REFERENCE_MODEL,1,N,0.948918,0.956365,0.952627,90341
1,REFERENCE_MODEL,1,S,0.015385,0.000360,0.000703,2781
2,REFERENCE_MODEL,1,V,0.610891,0.853507,0.712101,7229
3,REFERENCE_MODEL,1,F,0.000000,0.000000,0.000000,802
4,REFERENCE_MODEL,1,Q,0.996162,0.866461,0.926796,3894
5,REFERENCE_MODEL,2,N,0.949499,0.965453,0.957409,90341
6,REFERENCE_MODEL,2,S,0.000000,0.000000,0.000000,2781
7,REFERENCE_MODEL,2,V,0.488591,0.855997,0.622097,7229
8,REFERENCE_MODEL,2,F,0.002404,0.001247,0.001642,802
9,REFERENCE_MODEL,2,Q,0.000000,0.000000,0.000000,3894



CONFUSION MATRICES


,Actual_Class,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q,Repeat
0,Actual_N,86399,63,3469,398,12,1
1,Actual_S,2629,1,139,12,0,1
2,Actual_V,1022,1,6170,35,1,1
3,Actual_F,563,0,239,0,0,1
4,Actual_Q,437,0,83,0,3374,1
5,Actual_N,87220,96,2661,360,4,2
6,Actual_S,2641,0,133,7,0,2
7,Actual_V,986,6,6188,48,1,2
8,Actual_F,561,0,240,1,0,2
9,Actual_Q,451,0,3443,0,0,2



CELL 6 COMPLETE


In [9]:
# ============================================================
# CELL 7: RECORD-LEVEL BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def record_bootstrap_ci(
    pred,
    metric="Macro_F1_5_Class",
    n_boot=BOOTSTRAP_SAMPLES,
    seed=BOOTSTRAP_SEED
):

    rng = np.random.default_rng(seed)

    record_list = np.array(
        sorted(pred["Record_ID"].unique())
    )

    repeats = sorted(
        pred["Repeat"].unique()
    )

    # Store one prediction block per repeat and record.
    blocks = {
        (repeat, record): group
        for (repeat, record), group
        in pred.groupby(["Repeat", "Record_ID"])
    }

    bootstrap_scores = []

    for _ in range(n_boot):

        sampled_records = rng.choice(
            record_list,
            size=len(record_list),
            replace=True
        )

        repeat_scores = []

        for repeat in repeats:

            sample = pd.concat([
                blocks[(repeat, record)]
                for record in sampled_records
            ], ignore_index=True)

            yt = sample["True_Label"].to_numpy()
            yp = sample["Predicted_Label"].to_numpy()

            if metric == "Macro_F1_5_Class":

                score = f1_score(
                    yt, yp,
                    labels=CLASS_ORDER,
                    average="macro",
                    zero_division=0
                )

            elif metric == "Macro_F1_4_Class_N_S_V_Q":

                keep = np.isin(
                    yt,
                    FOUR_CLASS_ORDER
                )

                score = f1_score(
                    yt[keep],
                    yp[keep],
                    labels=FOUR_CLASS_ORDER,
                    average="macro",
                    zero_division=0
                )

            else:
                raise ValueError(f"Unknown metric: {metric}")

            repeat_scores.append(score)

        bootstrap_scores.append(
            np.mean(repeat_scores)
        )


    # Point estimate from the five original pooled repeats.
    point_scores = []

    for repeat in repeats:

        sample = pred[
            pred["Repeat"] == repeat
        ]

        yt = sample["True_Label"].to_numpy()
        yp = sample["Predicted_Label"].to_numpy()

        if metric == "Macro_F1_5_Class":

            score = f1_score(
                yt, yp,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0
            )

        else:

            keep = np.isin(
                yt,
                FOUR_CLASS_ORDER
            )

            score = f1_score(
                yt[keep],
                yp[keep],
                labels=FOUR_CLASS_ORDER,
                average="macro",
                zero_division=0
            )

        point_scores.append(score)


    lower, upper = np.quantile(
        bootstrap_scores,
        [0.025, 0.975]
    )

    return {
        "Metric": metric,
        "Point_Estimate": float(np.mean(point_scores)),
        "CI_95_Lower": float(lower),
        "CI_95_Upper": float(upper),
        "Bootstrap_Samples": n_boot,
        "Resampling_Unit": "Record_ID",
        "Bootstrap_Seed": seed
    }


# ------------------------------------------------------------
# Calculate both confidence intervals
# ------------------------------------------------------------

ci_results = pd.DataFrame([

    record_bootstrap_ci(
        predictions,
        "Macro_F1_5_Class"
    ),

    record_bootstrap_ci(
        predictions,
        "Macro_F1_4_Class_N_S_V_Q"
    )

])

ci_results.insert(
    0,
    "Experiment",
    EXPERIMENT
)

print("RECORD-LEVEL BOOTSTRAP RESULTS")
display(ci_results)

ci_results.to_csv(
    OUTPUT_DIR /
    f"m0_corrected_record_bootstrap_ci_{EXPERIMENT}.csv",
    index=False
)

print("\nCELL 7 COMPLETE")

RECORD-LEVEL BOOTSTRAP RESULTS


,Experiment,Metric,Point_Estimate,CI_95_Lower,CI_95_Upper,Bootstrap_Samples,Resampling_Unit,Bootstrap_Seed
0,REFERENCE_MODEL,Macro_F1_5_Class,0.435270,0.311816,0.467416,1000,Record_ID,20261105
1,REFERENCE_MODEL,Macro_F1_4_Class_N_S_V_Q,0.546823,0.393360,0.588774,1000,Record_ID,20261105



CELL 7 COMPLETE


In [11]:
# ============================================================
# FIX: CALCULATE CORRECTED DATASET SHA-256
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


corrected_dataset_sha256 = sha256_file(corrected_path)

print("Corrected dataset:", corrected_path)
print("SHA-256:", corrected_dataset_sha256)
print("PASS: Corrected dataset fingerprint calculated.")

Corrected dataset: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/MITBIH_MLII_DualLead_Dataset_corrected.csv
SHA-256: 47ff112ed63df5be463b8ba7b5823d86f77a3b61444e6b8ace852bbb1cdbd750
PASS: Corrected dataset fingerprint calculated.


In [12]:
# ============================================================
# CELL 8: FINAL VALIDATION AND MANIFEST
# ============================================================

checks = {

    "Corrected_dataset_selected": (
        corrected_path.name
        == "MITBIH_MLII_DualLead_Dataset_corrected.csv"
    ),

    "105047_valid_beats": (
        len(aami_df) == EXPECTED_BEATS
    ),

    "Expected_class_counts": (
        observed_counts == EXPECTED_COUNTS
    ),

    "46_records_45_people": (
        aami_df["Record_ID"].nunique() == EXPECTED_RECORDS
        and
        aami_df["Person_ID"].nunique() == EXPECTED_PEOPLE
    ),

    "201_202_same_person": (
        set(
            aami_df.loc[
                aami_df["Record_ID"].isin([201, 202]),
                "Person_ID"
            ].unique()
        ) == {"person_201_202"}
    ),

    "Original_split_fingerprint_verified": (
        computed_fingerprint == EXPECTED_SPLIT_FINGERPRINT
    ),

    "Original_row_index_mapping_verified": (
        ROW_INDEX_MODE in [
            "Created before AAMI filtering",
            "Created after AAMI filtering"
        ]
    ),

    "No_new_splits_generated": True,

    "25_fold_runs_completed": (
        len(fold_run_audit) == 25
    ),

    "OOF_predictions_complete": (
        len(predictions) == EXPECTED_BEATS * N_REPEATS
    ),

    "OOF_row_repeat_pairs_unique": (
        not predictions.duplicated(
            ["Repeat", "Row_Index"]
        ).any()
    ),

    "Finite_features": bool(
        np.isfinite(X).all()
    ),

    "Probability_rows_sum_to_one": bool(
        np.allclose(
            predictions[prob_cols].sum(axis=1),
            1.0,
            atol=1e-5
        )
    )

}


check_table = pd.DataFrame([
    {
        "Check": name,
        "Result": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])

print("=" * 70)
print("FINAL CORRECTED M0 VALIDATION")
print("=" * 70)

display(check_table)

check_table.to_csv(
    OUTPUT_DIR /
    f"m0_corrected_final_checks_{EXPERIMENT}.csv",
    index=False
)


if not all(checks.values()):

    raise AssertionError(
        "Corrected M0 validation failed. Stop and investigate."
    )


# ------------------------------------------------------------
# Save reproducibility manifest
# ------------------------------------------------------------

manifest = {

    "notebook": "02_M0_CORRECTED_BASELINE",

    "account_role": ACCOUNT_ROLE,

    "experiment": EXPERIMENT,

    "corrected_dataset": str(corrected_path),

    "corrected_dataset_sha256": corrected_dataset_sha256,

    "valid_beats": int(len(aami_df)),

    "records": int(aami_df["Record_ID"].nunique()),

    "people": int(aami_df["Person_ID"].nunique()),

    "class_counts": {
        k: int(v)
        for k, v in observed_counts.items()
    },

    "seed": SEED,

    "folds": N_SPLITS,

    "repeats": N_REPEATS,

    "row_index_mode": ROW_INDEX_MODE,

    "split_fingerprint": computed_fingerprint,

    "split_source": "Original saved M0 split files",

    "splits_regenerated": False,

    "bootstrap_samples": BOOTSTRAP_SAMPLES,

    "bootstrap_seed": BOOTSTRAP_SEED,

    "bootstrap_unit": "Record_ID",

    "random_forest_trees": TREES,

    "random_forest_max_features": MAX_FEATURES,

    "class_weight": None,

    "feature_count": int(X.shape[1]),

    "headline_metrics": headline.iloc[0].to_dict(),

    "confidence_intervals": ci_results.to_dict(
        orient="records"
    ),

    "checks": checks

}


manifest_path = (
    OUTPUT_DIR /
    f"m0_corrected_manifest_{EXPERIMENT}.json"
)

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRECTED M0 BASELINE COMPLETED")
print("=" * 70)

print("Experiment:", EXPERIMENT)

print(
    "Five-class Macro-F1:",
    round(
        float(headline.iloc[0]["Headline_Macro_F1_5_Class"]),
        6
    )
)

print(
    "Four-class Macro-F1:",
    round(
        float(headline.iloc[0]["Macro_F1_4_Class_N_S_V_Q"]),
        6
    )
)

print("Original split fingerprint:", computed_fingerprint)

print("Row_Index mode:", ROW_INDEX_MODE)

print("Saved results:", OUTPUT_DIR)

print("\nHistorical M0 outputs were not overwritten.")
print("No new splits were generated.")
print("No people were excluded.")

print("\nCORRECTED M0 RUN COMPLETE.")

FINAL CORRECTED M0 VALIDATION


,Check,Result
0,Corrected_dataset_selected,PASS
1,105047_valid_beats,PASS
2,Expected_class_counts,PASS
3,46_records_45_people,PASS
4,201_202_same_person,PASS
5,Original_split_fingerprint_verified,PASS
6,Original_row_index_mapping_verified,PASS
7,No_new_splits_generated,PASS
8,25_fold_runs_completed,PASS
9,OOF_predictions_complete,PASS



CORRECTED M0 BASELINE COMPLETED
Experiment: REFERENCE_MODEL
Five-class Macro-F1: 0.43527
Four-class Macro-F1: 0.546823
Original split fingerprint: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
Row_Index mode: Created after AAMI filtering
Saved results: /kaggle/working/ECG_Project/02_M0_Corrected/REFERENCE_MODEL

Historical M0 outputs were not overwritten.
No new splits were generated.
No people were excluded.

CORRECTED M0 RUN COMPLETE.
